## Set up imports + project root

In [1]:
# %% [markdown]
# # Match Prediction
# Use trained models to predict the outcome of a single match.

# %%
import os
from pathlib import Path
import sys
import pandas as pd
import joblib

# Set project root
os.chdir(r"c:\Github\fifa2026-prediction")
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: c:\Github\fifa2026-prediction


## Load trained models

In [2]:
# %% [markdown]
# ## Load Models

# %%
models_dir = PROJECT_ROOT / "models"

log_reg = joblib.load(models_dir / "logistic_regression.pkl")
scaler = joblib.load(models_dir / "scaler.pkl")
xgb = joblib.load(models_dir / "xgboost.pkl")

print("✅ Models loaded successfully.")


✅ Models loaded successfully.


## Create a helper function to build feature rows

In [3]:
# %% [markdown]
# ## Feature Builder for a Single Match

# %%
def build_match_features(home_team, away_team, home_elo, away_elo,
                         is_home_host=0, is_away_host=0,
                         is_neutral=0, is_friendly=0):

    df = pd.DataFrame([{
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_diff": home_elo - away_elo,
        "is_home_host": is_home_host,
        "is_away_host": is_away_host,
        "is_neutral": is_neutral,
        "is_friendly": is_friendly,
    }])

    return df


## Create a prediction function

In [4]:
# %% [markdown]
# ## Prediction Function

# %%
def predict_match(home_team, away_team, home_elo, away_elo,
                  is_home_host=0, is_away_host=0,
                  is_neutral=0, is_friendly=0):

    features = build_match_features(
        home_team, away_team, home_elo, away_elo,
        is_home_host, is_away_host, is_neutral, is_friendly
    )

    # Scale for logistic regression
    features_scaled = scaler.transform(features)

    # Predictions
    lr_pred = log_reg.predict(features_scaled)[0]
    xgb_pred = xgb.predict(features)[0]

    return {
        "home_team": home_team,
        "away_team": away_team,
        "logistic_regression_prediction": "Home Win" if lr_pred == 1 else "Not Home Win",
        "xgboost_prediction": "Home Win" if xgb_pred == 1 else "Not Home Win",
    }


## Test with a sample match

In [5]:
# %% [markdown]
# ## Example Prediction

# %%
result = predict_match(
    home_team="USA",
    away_team="Mexico",
    home_elo=1650,
    away_elo=1700,
    is_home_host=1,
    is_away_host=0,
    is_neutral=0,
    is_friendly=0
)

result


{'home_team': 'USA',
 'away_team': 'Mexico',
 'logistic_regression_prediction': 'Not Home Win',
 'xgboost_prediction': 'Home Win'}